# LawRAG Kaggle Offline Submission Notebook

This notebook is designed for Kaggle Code Competition constraints:
- Offline runtime (no internet)
- Deterministic output of `submission.csv`
- Bounded compute (target under 12 hours)

It includes the full implemented dense retrieval workflow:
1. Corpus unification and citation deduplication
2. Optional chunking
3. Dense retrieval baseline (FAISS + multilingual embeddings)
4. Optional cross-encoder reranker (if local models are available)
5. Citation-level aggregation
6. Evaluation on train/val (optional)
7. Submission generation for test set

In [19]:
import os
import re
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Iterable, Set, TypedDict
from uuid import uuid4

import numpy as np
import pandas as pd
from langchain_core.documents import Document
from langgraph.graph import END, START, StateGraph

try:
    import mlflow
except ImportError:
    mlflow = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 140)
print('Environment ready.')

Environment ready.


In [20]:
@dataclass
class Config:
    # Kaggle typically mounts competition data under /kaggle/input/<competition-name>/
    # For local testing, keep DATA_DIR='data'.
    DATA_DIR: str = 'data'

    # Reproducible artifact locations
    ARTIFACT_DIR: str = 'artifacts'
    SUBMISSION_NAME: str = 'submission.csv'

    # Corpus build controls
    # Default True to keep notebook stable under constrained memory.
    RESTRICT_TO_LABELED_CITATIONS: bool = True
    ENABLE_CHUNKING: bool = False
    CHUNK_CHARS: int = 1200
    OVERLAP_CHARS: int = 200

    # Dense retrieval stack
    MODE: str = 'dense'
    OUT_K: int = 12
    STAGE_K: int = 150
    CITATION_SCORE_AGG: str = 'max'  # 'max' or 'mean'

    # Optional reranker
    USE_RERANKER: bool = True
    RERANKER_TOP_N: int = 50

    # Offline-safe model configuration
    # Dense model must be present locally for strict offline mode.
    DENSE_MODEL_NAME: str = 'intfloat/multilingual-e5-base'
    RERANKER_MODEL_NAME: str = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

    # Optional local model paths from Kaggle datasets, e.g. '/kaggle/input/my-models/e5-base'
    LOCAL_DENSE_MODEL_PATH: Optional[str] = None
    LOCAL_RERANKER_MODEL_PATH: Optional[str] = None

    # MLflow experiment tracking
    ENABLE_MLFLOW: bool = True
    MLFLOW_TRACKING_URI: Optional[str] = 'http://127.0.0.1:5000'
    MLFLOW_EXPERIMENT: str = 'lawrag-dense-notebook'

cfg = Config()

data_dir = Path(cfg.DATA_DIR)
artifact_dir = Path(cfg.ARTIFACT_DIR)
artifact_dir.mkdir(parents=True, exist_ok=True)

print(cfg)
print('Data dir exists:', data_dir.exists())

Config(DATA_DIR='data', ARTIFACT_DIR='artifacts', SUBMISSION_NAME='submission.csv', RESTRICT_TO_LABELED_CITATIONS=True, ENABLE_CHUNKING=False, CHUNK_CHARS=1200, OVERLAP_CHARS=200, MODE='dense', OUT_K=12, STAGE_K=150, CITATION_SCORE_AGG='max', USE_RERANKER=True, RERANKER_TOP_N=50, DENSE_MODEL_NAME='intfloat/multilingual-e5-base', RERANKER_MODEL_NAME='cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', LOCAL_DENSE_MODEL_PATH=None, LOCAL_RERANKER_MODEL_PATH=None, ENABLE_MLFLOW=True, MLFLOW_TRACKING_URI='http://127.0.0.1:5000', MLFLOW_EXPERIMENT='lawrag-dense-notebook')
Data dir exists: True


In [21]:
MULTISPACE_RE = re.compile(r'\s+')

def normalize_text(text: str) -> str:
    """Normalize text by collapsing whitespace and stripping."""
    if text is None:
        return ''
    return MULTISPACE_RE.sub(' ', str(text)).strip()

def split_citations(raw: str) -> List[str]:
    """Split a raw citation string into a list of normalized citations."""
    if raw is None:
        return []
    parts = [normalize_text(p) for p in str(raw).split(';')]
    return [p for p in parts if p]

def join_citations(citations: Sequence[str]) -> str:
    """Join a list of citations into a single normalized string."""
    seen = set()
    out = []
    for cit in citations:
        c = normalize_text(cit)
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    return ';'.join(out)

def unique_preserve_order(items: Iterable[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out

In [22]:
@dataclass
class DataPaths:
    root: Path

    @property
    def train(self) -> Path:
        return self.root / 'train.csv'

    @property
    def val(self) -> Path:
        return self.root / 'val.csv'

    @property
    def test(self) -> Path:
        return self.root / 'test.csv'

    @property
    def laws(self) -> Path:
        return self.root / 'laws_de.csv'

    @property
    def court(self) -> Path:
        return self.root / 'court_considerations.csv'

def _ensure_columns(df: pd.DataFrame, defaults: Dict[str, str]) -> pd.DataFrame:
    """
    Ensure that the DataFrame contains the specified columns.
    If a column is missing, it will be added with the default value.
    """
    out = df.copy()
    for col, default_val in defaults.items():
        if col not in out.columns:
            out[col] = default_val
    return out

def load_labeled_citation_set(paths: DataPaths) -> Set[str]:
    """
    Load the set of labeled citations from the training and validation datasets.
    """

    labeled = set()
    for p in [paths.train, paths.val]:
        if not p.exists():
            continue
        df = pd.read_csv(p, usecols=['gold_citations'], dtype=str)
        for raw in df['gold_citations'].fillna(''):
            labeled.update(split_citations(raw))
    return labeled

def build_unified_corpus(paths: DataPaths, allowed_citations: Optional[Set[str]] = None) -> pd.DataFrame:
    """
    Build a unified corpus from the laws and court considerations datasets.
    """
    best_by_citation: Dict[str, Dict[str, object]] = {}

    def consume_frame(df: pd.DataFrame, source: str) -> None:
        """
        Process a DataFrame and update the best_by_citation dictionary.
        """
        local = _ensure_columns(df, {'title': ''})
        for row in local.itertuples(index=False):
            citation = normalize_text(getattr(row, 'citation', ''))
            if not citation:
                continue
            if allowed_citations is not None and citation not in allowed_citations:
                continue

            text = normalize_text(getattr(row, 'text', ''))
            title = normalize_text(getattr(row, 'title', ''))
            full_text = normalize_text(f'{title}\n{text}')
            length = len(full_text)

            prev = best_by_citation.get(citation)
            if prev is None or length > prev['length']:
                best_by_citation[citation] = {
                    'doc_id': f'{citation}__chunk0',
                    'citation': citation,
                    'text': text,
                    'title': title,
                    'source': source,
                    'full_text': full_text,
                    'length': length,
                }

    laws = pd.read_csv(paths.laws, usecols=['citation', 'text', 'title'], dtype=str)
    consume_frame(laws, source='law')

    for chunk in pd.read_csv(paths.court, usecols=['citation', 'text'], dtype=str, chunksize=200_000):
        consume_frame(chunk, source='court')

    corpus = pd.DataFrame(best_by_citation.values()).sort_values('citation').reset_index(drop=True)
    return corpus

def chunk_corpus(corpus_df: pd.DataFrame, chunk_chars: int = 1200, overlap_chars: int = 200) -> pd.DataFrame:
    rows = []
    stride = max(1, chunk_chars - overlap_chars)

    for rec in corpus_df.to_dict('records'):
        text = rec['full_text']
        if len(text) <= chunk_chars:
            rows.append(rec)
            continue

        chunk_idx = 0
        for start in range(0, len(text), stride):
            piece = text[start:start + chunk_chars]
            if len(piece) < 80:
                continue
            new_row = dict(rec)
            new_row['doc_id'] = f"{rec['citation']}__chunk{chunk_idx}"
            new_row['full_text'] = piece
            new_row['length'] = len(piece)
            rows.append(new_row)
            chunk_idx += 1

    return pd.DataFrame(rows)

In [23]:
from langchain_core.embeddings import Embeddings

def _is_local_adapter_dir(model_name_or_path: str) -> bool:
    model_path = Path(model_name_or_path)
    return model_path.is_dir() and (model_path / 'adapter_config.json').exists()

def _get_base_model_from_adapter(adapter_dir: str) -> Optional[str]:
    cfg_path = Path(adapter_dir) / 'adapter_config.json'
    if not cfg_path.exists():
        return None
    import json
    with cfg_path.open('r', encoding='utf-8') as f:
        cfg = json.load(f)
    base_ref = cfg.get('base_model_name_or_path')
    return str(base_ref) if base_ref else None

def load_sentence_transformer(model_name_or_path: str):
    from sentence_transformers import SentenceTransformer

    if not _is_local_adapter_dir(model_name_or_path):
        return SentenceTransformer(model_name_or_path)

    base_ref = _get_base_model_from_adapter(model_name_or_path)
    if not base_ref:
        return SentenceTransformer(model_name_or_path)

    from peft import PeftModel

    model = SentenceTransformer(base_ref)
    transformer_module = model._first_module()
    auto_model = getattr(transformer_module, 'auto_model', None)
    if auto_model is None:
        return SentenceTransformer(model_name_or_path)

    peft_model = PeftModel.from_pretrained(auto_model, model_name_or_path)
    if hasattr(peft_model, 'merge_and_unload'):
        transformer_module.auto_model = peft_model.merge_and_unload()
    else:
        transformer_module.auto_model = peft_model

    return model

def load_cross_encoder(model_name_or_path: str):
    from sentence_transformers import CrossEncoder

    if not _is_local_adapter_dir(model_name_or_path):
        return CrossEncoder(model_name_or_path)

    base_ref = _get_base_model_from_adapter(model_name_or_path)
    if not base_ref:
        return CrossEncoder(model_name_or_path)

    from peft import PeftModel

    model = CrossEncoder(base_ref)
    peft_model = PeftModel.from_pretrained(model.model, model_name_or_path)
    if hasattr(peft_model, 'merge_and_unload'):
        model.model = peft_model.merge_and_unload()
    else:
        model.model = peft_model
    return model

@dataclass
class RetrievalResult:
    citation: str
    doc_id: str
    score: float
    source: str
    text: str

class FaissCitationRetriever:
    class _SentenceTransformerEmbeddings(Embeddings):
        def __init__(self, model_name_or_path: str):
            self.model = load_sentence_transformer(model_name_or_path)

        def embed_documents(self, texts: List[str]) -> List[List[float]]:
            emb = self.model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            return emb.tolist()

        def embed_query(self, text: str) -> List[float]:
            emb = self.model.encode(
                [text],
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            return emb[0].tolist()

    def __init__(self, corpus_df: pd.DataFrame, model_name_or_path: str):
        from langchain_community.vectorstores import FAISS

        self.corpus = corpus_df.reset_index(drop=True)
        self.embedding = self._SentenceTransformerEmbeddings(model_name_or_path)

        docs = [
            Document(
                page_content=row.get('full_text', '') or '',
                metadata={
                    'citation': row['citation'],
                    'doc_id': row['doc_id'],
                    'source': row.get('source', ''),
                },
            )
            for row in self.corpus.to_dict('records')
        ]

        self.vectorstore = FAISS.from_documents(
            documents=docs,
            embedding=self.embedding,
        )

    @staticmethod
    def _distance_to_similarity(distance: float) -> float:
        return 1.0 / (1.0 + max(float(distance), 0.0))

    def search(self, query: str, topk: int = 20) -> List[RetrievalResult]:
        out = []
        docs_and_dist = self.vectorstore.similarity_search_with_score(query, k=topk)
        for doc, dist in docs_and_dist:
            out.append(
                RetrievalResult(
                    citation=doc.metadata.get('citation', ''),
                    doc_id=doc.metadata.get('doc_id', ''),
                    score=self._distance_to_similarity(float(dist)),
                    source=doc.metadata.get('source', ''),
                    text=doc.page_content,
                )
            )
        return out

def aggregate_results_by_citation(results: List[RetrievalResult], score_agg: str = 'max', topk: Optional[int] = None) -> List[RetrievalResult]:
    grouped: Dict[str, List[RetrievalResult]] = {}
    for rec in results:
        grouped.setdefault(rec.citation, []).append(rec)

    aggregated = []
    for citation, recs in grouped.items():
        best_rec = max(recs, key=lambda r: r.score)
        if score_agg == 'mean':
            score = float(sum(r.score for r in recs) / len(recs))
        else:
            score = float(best_rec.score)

        aggregated.append(
            RetrievalResult(
                citation=citation,
                doc_id=best_rec.doc_id,
                score=score,
                source=best_rec.source,
                text=best_rec.text,
            )
        )

    aggregated.sort(key=lambda r: r.score, reverse=True)
    if topk is None:
        return aggregated
    return aggregated[:topk]

class CrossEncoderCitationReranker:
    def __init__(self, model_name_or_path: str):
        self.model = load_cross_encoder(model_name_or_path)

    def rerank(self, query: str, candidates: List[RetrievalResult], batch_size: int = 32) -> List[RetrievalResult]:
        if not candidates:
            return []

        pairs = [(query, c.text) for c in candidates]
        scores = self.model.predict(pairs, batch_size=batch_size, show_progress_bar=False)

        rescored = []
        for cand, score in zip(candidates, scores):
            rescored.append(
                RetrievalResult(
                    citation=cand.citation,
                    doc_id=cand.doc_id,
                    score=float(score),
                    source=cand.source,
                    text=cand.text,
                )
            )

        rescored.sort(key=lambda r: r.score, reverse=True)
        return rescored

In [24]:
@dataclass
class PipelineConfig:
    mode: str = 'dense'
    dense_model_name: str = 'intfloat/multilingual-e5-base'
    citation_score_agg: str = 'max'
    stage_k: int = 150
    out_k: int = 12
    use_reranker: bool = True
    reranker_model_name: str = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
    reranker_top_n: int = 50
    reranker_batch_size: int = 32

class PipelineState(TypedDict, total=False):
    query: str
    stage_results: List[RetrievalResult]
    aggregated: List[RetrievalResult]
    final_results: List[RetrievalResult]

class CitationRAGPipeline:
    def __init__(self, corpus_df: pd.DataFrame, config: PipelineConfig, local_dense_model_path: Optional[str] = None, local_reranker_model_path: Optional[str] = None):
        self.config = config

        dense_model_ref = local_dense_model_path or self.config.dense_model_name
        reranker_model_ref = local_reranker_model_path or self.config.reranker_model_name

        self.dense = FaissCitationRetriever(corpus_df, dense_model_ref)

        self.reranker = None
        if self.config.use_reranker:
            self.reranker = CrossEncoderCitationReranker(reranker_model_ref)

        graph = StateGraph(PipelineState)
        graph.add_node('retrieve_stage', self._retrieve_stage)
        graph.add_node('aggregate', self._aggregate)
        graph.add_node('rerank_or_finalize', self._rerank_or_finalize)
        graph.add_edge(START, 'retrieve_stage')
        graph.add_edge('retrieve_stage', 'aggregate')
        graph.add_edge('aggregate', 'rerank_or_finalize')
        graph.add_edge('rerank_or_finalize', END)
        self.graph = graph.compile()

    def _retrieve_stage(self, state: PipelineState) -> PipelineState:
        query = state['query']
        stage_results = self.dense.search(query, topk=self.config.stage_k)
        return {'stage_results': stage_results}

    def _aggregate(self, state: PipelineState) -> PipelineState:
        aggregated = aggregate_results_by_citation(
            state.get('stage_results', []),
            score_agg=self.config.citation_score_agg,
            topk=self.config.stage_k,
        )
        return {'aggregated': aggregated}

    def _rerank_or_finalize(self, state: PipelineState) -> PipelineState:
        aggregated = state.get('aggregated', [])
        if self.reranker is None:
            return {'final_results': aggregated[:self.config.out_k]}

        query = state['query']
        top_n = min(self.config.reranker_top_n, len(aggregated))
        head = aggregated[:top_n]
        tail = aggregated[top_n:]
        reranked_head = self.reranker.rerank(query, head, batch_size=self.config.reranker_batch_size)

        return {'final_results': (reranked_head + tail)[:self.config.out_k]}

    def retrieve(self, query: str) -> List[RetrievalResult]:
        out = self.graph.invoke({'query': query})
        return out.get('final_results', [])

    def predict(self, query: str) -> str:
        results = self.retrieve(query)
        citations = unique_preserve_order([r.citation for r in results])
        return join_citations(citations[:self.config.out_k])

def f1_for_sets(gold: Set[str], pred: Set[str]) -> float:
    if not gold and not pred:
        return 1.0
    if not pred:
        return 0.0

    tp = len(gold & pred)
    precision = tp / max(len(pred), 1)
    recall = tp / max(len(gold), 1)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def macro_f1(gold_list: Sequence[str], pred_list: Sequence[str]) -> float:
    assert len(gold_list) == len(pred_list)
    scores = []
    for g_raw, p_raw in zip(gold_list, pred_list):
        g = set(split_citations(g_raw))
        p = set(split_citations(p_raw))
        scores.append(f1_for_sets(g, p))
    return float(sum(scores) / len(scores)) if scores else 0.0

In [25]:
paths = DataPaths(root=data_dir)

allowed = load_labeled_citation_set(paths) if cfg.RESTRICT_TO_LABELED_CITATIONS else None
corpus_df = build_unified_corpus(paths, allowed_citations=allowed)

if cfg.ENABLE_CHUNKING:
    corpus_df = chunk_corpus(corpus_df, chunk_chars=cfg.CHUNK_CHARS, overlap_chars=cfg.OVERLAP_CHARS)

print('Corpus rows:', len(corpus_df))
print('Unique citations:', corpus_df['citation'].nunique())

# Save corpus artifact for reproducibility
corpus_path = artifact_dir / ('corpus_labeled.parquet' if cfg.RESTRICT_TO_LABELED_CITATIONS else 'corpus.parquet')
corpus_df.to_parquet(corpus_path, index=False)
print('Saved corpus:', corpus_path)

Corpus rows: 2111
Unique citations: 2111
Saved corpus: artifacts/corpus_labeled.parquet



## Fine-Tuning: Bi-Encoder (Dense) and Cross-Encoder (Reranker)

Supervised fine-tuning of both retrieval models on the labeled training set using the `SentenceTransformerTrainer` / `CrossEncoderTrainer` APIs from sentence-transformers ≥ 3.

| Model | Loss | Input |
|---|---|---|
| `multilingual-e5-base` (bi-encoder) | `MultipleNegativesRankingLoss` | (query, positive passage) pairs with in-batch negatives |
| `mmarco-mMiniLMv2` (cross-encoder) | `BinaryCrossEntropyLoss` | (query, passage, 0\|1) triples with random hard negatives |

Fine-tuned models are saved under `artifacts/models/` and automatically wired into `cfg.LOCAL_DENSE_MODEL_PATH` / `cfg.LOCAL_RERANKER_MODEL_PATH` for the pipeline cells below.

Set `FinetuneConfig.run_dense_finetune = False` or `run_reranker_finetune = False` to skip a stage (e.g. for inference-only Kaggle runs with pre-trained saved weights).


In [26]:

@dataclass
class FinetuneConfig:
    """Hyperparameters for SFT of the dense model and reranker."""
    # Training hyperparameters
    epochs: int = 3
    batch_size: int = 16
    learning_rate: float = 2e-5
    warmup_ratio: float = 0.1

    # Mid-epoch evaluation/checkpoint cadence
    eval_steps: int = 100
    save_steps: int = 100
    save_total_limit: int = 2

    # LoRA adapter settings
    use_lora: bool = True
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_bias: str = 'none'
    lora_target_modules: List[str] = field(default_factory=lambda: ["query", "key", "value", "dense"])

    # Number of random hard-negatives per query for the cross-encoder
    negatives_per_query: int = 3

    # Output directories (also set on cfg.LOCAL_*_MODEL_PATH so the pipeline picks them up)
    dense_output_dir: str = 'artifacts/models/dense_ft'
    reranker_output_dir: str = 'artifacts/models/reranker_ft'

    # Toggle training stages (set False to run inference-only)
    run_dense_finetune: bool = True
    run_reranker_finetune: bool = True


ft_cfg = FinetuneConfig()
print(ft_cfg)


FinetuneConfig(epochs=3, batch_size=16, learning_rate=2e-05, warmup_ratio=0.1, eval_steps=100, save_steps=100, save_total_limit=2, use_lora=True, lora_r=32, lora_alpha=64, lora_dropout=0.05, lora_bias='none', lora_target_modules=['query', 'key', 'value', 'dense'], negatives_per_query=3, dense_output_dir='artifacts/models/dense_ft', reranker_output_dir='artifacts/models/reranker_ft', run_dense_finetune=True, run_reranker_finetune=True)


In [27]:

# -- Build train/validation pairs for SFT ------------------------------------
def build_finetune_pairs(
    train_df: pd.DataFrame,
    corpus_df: pd.DataFrame,
    negatives_per_query: int = 3,
    seed: int = SEED,
) -> tuple[list[dict], list[dict]]:
    """Return (bi_encoder_pairs, cross_encoder_pairs).

    Bi-encoder  - columns: anchor, positive  (for MultipleNegativesRankingLoss)
    Cross-encoder - columns: sentence1, sentence2, label  (for BinaryCrossEntropyLoss)

    E5 input format requires 'query: ...' / 'passage: ...' prefixes for bi-encoder.
    Hard negatives for the cross-encoder are sampled randomly from the corpus,
    excluding the query's gold citations.
    """
    rng = random.Random(seed)
    citation_to_text: dict[str, str] = dict(
        zip(corpus_df['citation'].tolist(), corpus_df['full_text'].tolist())
    )
    all_citations = corpus_df['citation'].tolist()

    bi_encoder_pairs: list[dict] = []
    cross_encoder_pairs: list[dict] = []

    for row in train_df.itertuples(index=False):
        query: str = normalize_text(getattr(row, 'query', ''))
        gold_cits: list[str] = split_citations(getattr(row, 'gold_citations', ''))

        positives: list[str] = [
            citation_to_text[c] for c in gold_cits if c in citation_to_text
        ]
        if not positives:
            continue

        # Bi-encoder pairs (one per positive)
        for pos_text in positives:
            bi_encoder_pairs.append(
                {'anchor': f'query: {query}', 'positive': f'passage: {pos_text}'}
            )

        # Cross-encoder pairs: positives + random hard negatives
        gold_set: set[str] = set(gold_cits)
        neg_pool = [c for c in all_citations if c not in gold_set]
        neg_sample = rng.sample(neg_pool, min(negatives_per_query, len(neg_pool)))

        for pos_text in positives:
            cross_encoder_pairs.append(
                {'sentence1': query, 'sentence2': pos_text, 'label': 1.0}
            )
        for neg_cit in neg_sample:
            neg_text = citation_to_text.get(neg_cit, '')
            if neg_text:
                cross_encoder_pairs.append(
                    {'sentence1': query, 'sentence2': neg_text, 'label': 0.0}
                )

    return bi_encoder_pairs, cross_encoder_pairs


_train_df = pd.read_csv(paths.train)
_val_df = pd.read_csv(paths.val)

bi_pairs, ce_pairs = build_finetune_pairs(
    _train_df, corpus_df, negatives_per_query=ft_cfg.negatives_per_query, seed=SEED
)
val_bi_pairs, val_ce_pairs = build_finetune_pairs(
    _val_df, corpus_df, negatives_per_query=ft_cfg.negatives_per_query, seed=SEED + 1
)

n_pos = sum(p['label'] == 1.0 for p in ce_pairs)
n_neg = sum(p['label'] == 0.0 for p in ce_pairs)
n_val_pos = sum(p['label'] == 1.0 for p in val_ce_pairs)
n_val_neg = sum(p['label'] == 0.0 for p in val_ce_pairs)

print(f'Train bi-encoder pairs   : {len(bi_pairs):,}')
print(f'Train cross-encoder pairs: {len(ce_pairs):,}  (pos={n_pos:,}  neg={n_neg:,})')
print(f'Val bi-encoder pairs     : {len(val_bi_pairs):,}')
print(f'Val cross-encoder pairs  : {len(val_ce_pairs):,}  (pos={n_val_pos:,}  neg={n_val_neg:,})')


Train bi-encoder pairs   : 3,319
Train cross-encoder pairs: 6,316  (pos=3,319  neg=2,997)
Val bi-encoder pairs     : 251
Val cross-encoder pairs  : 281  (pos=251  neg=30)


In [28]:

# -- Bi-encoder fine-tuning -----------------------------------------------
if ft_cfg.run_dense_finetune:
    from datasets import Dataset as HFDataset
    from peft import LoraConfig, TaskType, get_peft_model
    from sentence_transformers import (
        SentenceTransformer,
        SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments,
    )
    from sentence_transformers.losses import MultipleNegativesRankingLoss
    from sentence_transformers.training_args import BatchSamplers

    dense_model_ref = cfg.LOCAL_DENSE_MODEL_PATH or cfg.DENSE_MODEL_NAME
    bi_model = SentenceTransformer(dense_model_ref)

    if ft_cfg.use_lora:
        transformer_module = bi_model._first_module()
        base_auto_model = transformer_module.auto_model
        bi_lora_cfg = LoraConfig(
            r=ft_cfg.lora_r,
            lora_alpha=ft_cfg.lora_alpha,
            lora_dropout=ft_cfg.lora_dropout,
            bias=ft_cfg.lora_bias,
            target_modules=ft_cfg.lora_target_modules,
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        transformer_module.auto_model = get_peft_model(base_auto_model, bi_lora_cfg)
        print('LoRA enabled for bi-encoder:')
        transformer_module.auto_model.print_trainable_parameters()

    bi_dataset = HFDataset.from_list(bi_pairs)
    bi_eval_dataset = HFDataset.from_list(val_bi_pairs)
    bi_loss = MultipleNegativesRankingLoss(bi_model)

    bi_args = SentenceTransformerTrainingArguments(
        output_dir=ft_cfg.dense_output_dir,
        num_train_epochs=ft_cfg.epochs,
        per_device_train_batch_size=ft_cfg.batch_size,
        learning_rate=ft_cfg.learning_rate,
        warmup_ratio=ft_cfg.warmup_ratio,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        save_strategy='steps',
        save_steps=ft_cfg.save_steps,
        save_total_limit=ft_cfg.save_total_limit,
        eval_strategy='steps',
        eval_steps=ft_cfg.eval_steps,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        fp16=False,
        bf16=False,
        logging_steps=50,
        seed=SEED,
    )

    bi_trainer = SentenceTransformerTrainer(
        model=bi_model,
        args=bi_args,
        train_dataset=bi_dataset,
        eval_dataset=bi_eval_dataset,
        loss=bi_loss,
    )

    print('Fine-tuning bi-encoder ...')
    bi_trainer.train()

    # Note: load_best_model_at_end=True means the trainer already reloaded
    # the best checkpoint before train() returned, so bi_model has the best weights
    if bi_trainer.state.best_model_checkpoint:
        print('Best bi-encoder checkpoint:', bi_trainer.state.best_model_checkpoint)

    bi_model.save_pretrained(ft_cfg.dense_output_dir)
    print('Saved bi-encoder to:', ft_cfg.dense_output_dir)

    # Point pipeline to the fine-tuned model
    cfg.LOCAL_DENSE_MODEL_PATH = ft_cfg.dense_output_dir
else:
    print('Skipping bi-encoder fine-tuning (run_dense_finetune=False)')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3208.94it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


LoRA enabled for bi-encoder:
trainable params: 5,357,568 || all params: 283,401,216 || trainable%: 1.8905


Fine-tuning bi-encoder ...


Step,Training Loss,Validation Loss
100,1.702219,1.158077
200,1.021755,1.173841
300,1.027499,1.230213
400,0.932582,1.248244
500,0.847955,1.258039
600,0.851584,1.266316


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5537.09it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 0it [00:00, ?it/s]
XLMRobertaModel LOAD REPORT from: artifacts/models/dense_ft/checkpoint-100
Key                                                                                  | Status     | 
-------------------------------------------------------------------------------------+------------+-
base_model.model.encoder.layer.{0...11}.attention.self.value.lora_A.default.weight   | UNEXPECTED | 
base_model.model.encoder.layer.{0...11}.intermediate.dense.lora_B.default.weight     | UNEXPECTED | 
base_model.model.encoder.layer.{0...11}.attention.output.dense.lora_A.default.weight | UNEXPECTED | 
b

Best bi-encoder checkpoint: artifacts/models/dense_ft/checkpoint-100
Saved bi-encoder to: artifacts/models/dense_ft


In [29]:

# -- Cross-encoder fine-tuning --------------------------------------------
if ft_cfg.run_reranker_finetune:
    from datasets import Dataset as HFDataset
    from peft import LoraConfig, TaskType, get_peft_model
    from sentence_transformers.cross_encoder import (
        CrossEncoder,
        CrossEncoderTrainer,
        CrossEncoderTrainingArguments,
    )
    from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

    reranker_model_ref = cfg.LOCAL_RERANKER_MODEL_PATH or cfg.RERANKER_MODEL_NAME
    ce_model = CrossEncoder(reranker_model_ref, num_labels=1)

    if ft_cfg.use_lora:
        ce_lora_cfg = LoraConfig(
            r=ft_cfg.lora_r,
            lora_alpha=ft_cfg.lora_alpha,
            lora_dropout=ft_cfg.lora_dropout,
            bias=ft_cfg.lora_bias,
            target_modules=ft_cfg.lora_target_modules,
            task_type=TaskType.SEQ_CLS,
        )
        ce_model.model = get_peft_model(ce_model.model, ce_lora_cfg)
        print('LoRA enabled for cross-encoder:')
        ce_model.model.print_trainable_parameters()

    ce_dataset = HFDataset.from_list(ce_pairs)
    ce_eval_dataset = HFDataset.from_list(val_ce_pairs)
    ce_loss = BinaryCrossEntropyLoss(ce_model)

    ce_args = CrossEncoderTrainingArguments(
        output_dir=ft_cfg.reranker_output_dir,
        num_train_epochs=ft_cfg.epochs,
        per_device_train_batch_size=ft_cfg.batch_size,
        learning_rate=ft_cfg.learning_rate,
        warmup_ratio=ft_cfg.warmup_ratio,
        save_strategy='steps',
        save_steps=ft_cfg.save_steps,
        save_total_limit=ft_cfg.save_total_limit,
        eval_strategy='steps',
        eval_steps=ft_cfg.eval_steps,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        fp16=False,
        bf16=False,
        logging_steps=50,
        seed=SEED,
    )

    ce_trainer = CrossEncoderTrainer(
        model=ce_model,
        args=ce_args,
        train_dataset=ce_dataset,
        eval_dataset=ce_eval_dataset,
        loss=ce_loss,
    )

    print('Fine-tuning cross-encoder (reranker) ...')
    ce_trainer.train()

    # Note: load_best_model_at_end=True means the trainer already reloaded
    # the best checkpoint before train() returned, so ce_model has the best weights
    if ce_trainer.state.best_model_checkpoint:
        print('Best reranker checkpoint:', ce_trainer.state.best_model_checkpoint)

    ce_model.save_pretrained(ft_cfg.reranker_output_dir)
    print('Saved reranker to:', ft_cfg.reranker_output_dir)

    # Point pipeline to the fine-tuned model
    cfg.LOCAL_RERANKER_MODEL_PATH = ft_cfg.reranker_output_dir
else:
    print('Skipping reranker fine-tuning (run_reranker_finetune=False)')


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3311.50it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


LoRA enabled for cross-encoder:
trainable params: 2,802,433 || all params: 120,443,522 || trainable%: 2.3268


Token indices sequence length is longer than the specified maximum sequence length for this model (656 > 512). Running this sequence through the model will result in indexing errors


Fine-tuning cross-encoder (reranker) ...


Step,Training Loss,Validation Loss
100,1.028011,1.116642
200,0.690854,0.588028
300,0.650095,0.562923
400,0.660343,0.603115
500,0.611994,0.621029
600,0.578562,0.652865
700,0.574870,0.697751
800,0.590883,0.670817
900,0.549172,0.665800
1000,0.540148,0.694946


Best reranker checkpoint: artifacts/models/reranker_ft/checkpoint-300
Saved reranker to: artifacts/models/reranker_ft


In [30]:
dense_artifact_path = artifact_dir / 'models' / 'dense_ft'
reranker_artifact_path = artifact_dir / 'models' / 'reranker_ft'

if cfg.LOCAL_DENSE_MODEL_PATH is None and dense_artifact_path.exists():
    cfg.LOCAL_DENSE_MODEL_PATH = str(dense_artifact_path)
if cfg.LOCAL_RERANKER_MODEL_PATH is None and reranker_artifact_path.exists():
    cfg.LOCAL_RERANKER_MODEL_PATH = str(reranker_artifact_path)

pipeline_cfg = PipelineConfig(
    mode=cfg.MODE,
    dense_model_name=cfg.DENSE_MODEL_NAME,
    citation_score_agg=cfg.CITATION_SCORE_AGG,
    stage_k=cfg.STAGE_K,
    out_k=cfg.OUT_K,
    use_reranker=cfg.USE_RERANKER,
    reranker_model_name=cfg.RERANKER_MODEL_NAME,
    reranker_top_n=cfg.RERANKER_TOP_N,
    reranker_batch_size=32,

)

pipeline = CitationRAGPipeline(
    corpus_df=corpus_df,
    config=pipeline_cfg,
    local_dense_model_path=cfg.LOCAL_DENSE_MODEL_PATH,
    local_reranker_model_path=cfg.LOCAL_RERANKER_MODEL_PATH,
)

print('Pipeline initialized with mode:', pipeline_cfg.mode)
print('Dense model ref:', cfg.LOCAL_DENSE_MODEL_PATH or cfg.DENSE_MODEL_NAME)
print('Reranker model ref:', cfg.LOCAL_RERANKER_MODEL_PATH or cfg.RERANKER_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4303.74it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5059.38it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline initialized with mode: dense
Dense model ref: artifacts/models/dense_ft
Reranker model ref: artifacts/models/reranker_ft


In [31]:
# Disable MLflow for local/offline execution when no tracking server is running
cfg.ENABLE_MLFLOW = False
print('ENABLE_MLFLOW set to', cfg.ENABLE_MLFLOW)

ENABLE_MLFLOW set to False


In [32]:
# Optional local evaluation for sanity checks + MLflow logging
train_df = pd.read_csv(paths.train)
val_df = pd.read_csv(paths.val)

def evaluate(df: pd.DataFrame, name: str) -> float:
    preds = [pipeline.predict(q) for q in df['query'].fillna('').tolist()]
    score = macro_f1(df['gold_citations'].fillna('').tolist(), preds)
    print(f'{name} macro_f1: {score:.6f}')
    return score

if cfg.ENABLE_MLFLOW and mlflow is not None:
    if cfg.MLFLOW_TRACKING_URI:
        mlflow.set_tracking_uri(cfg.MLFLOW_TRACKING_URI)
    mlflow.set_experiment(cfg.MLFLOW_EXPERIMENT)
    with mlflow.start_run(run_name='dense_single_eval'):
        mlflow.log_params({
            'mode': cfg.MODE,
            'dense_model': cfg.DENSE_MODEL_NAME,
            'stage_k': cfg.STAGE_K,
            'out_k': cfg.OUT_K,
            'citation_score_agg': cfg.CITATION_SCORE_AGG,
            'use_reranker': cfg.USE_RERANKER,
            'reranker_model': cfg.RERANKER_MODEL_NAME,
            'reranker_top_n': cfg.RERANKER_TOP_N,
        })
        train_score = evaluate(train_df, 'train')
        val_score = evaluate(val_df, 'val')
        mlflow.log_metric('train_macro_f1', float(train_score))
        mlflow.log_metric('val_macro_f1', float(val_score))
else:
    if cfg.ENABLE_MLFLOW and mlflow is None:
        print('MLflow not installed; running evaluation without tracking.')
    evaluate(train_df, 'train')
    evaluate(val_df, 'val')

train macro_f1: 0.088647
val macro_f1: 0.118470


In [33]:
# Optional MLflow sweep for dense retriever
if cfg.ENABLE_MLFLOW and mlflow is not None:
    if cfg.MLFLOW_TRACKING_URI:
        mlflow.set_tracking_uri(cfg.MLFLOW_TRACKING_URI)
    mlflow.set_experiment(cfg.MLFLOW_EXPERIMENT)

    sweep_stage_k = [100, 150]
    sweep_out_k = [8, 12]
    sweep_score_agg = ['max', 'mean']

    best = None
    for stage_k in sweep_stage_k:
        for out_k in sweep_out_k:
            for score_agg in sweep_score_agg:
                sweep_cfg = PipelineConfig(
                    mode='dense',
                    dense_model_name=cfg.DENSE_MODEL_NAME,
                    citation_score_agg=score_agg,
                    stage_k=stage_k,
                    out_k=out_k,
                    use_reranker=cfg.USE_RERANKER,
                    reranker_model_name=cfg.RERANKER_MODEL_NAME,
                    reranker_top_n=cfg.RERANKER_TOP_N,
                )
                sweep_pipeline = CitationRAGPipeline(
                    corpus_df=corpus_df,
                    config=sweep_cfg,
                    local_dense_model_path=cfg.LOCAL_DENSE_MODEL_PATH,
                    local_reranker_model_path=cfg.LOCAL_RERANKER_MODEL_PATH,
                )

                run_name = f'dense_sweep_{stage_k}_{out_k}_{score_agg}'
                with mlflow.start_run(run_name=run_name):
                    mlflow.log_params({
                        'mode': 'dense',
                        'dense_model': cfg.DENSE_MODEL_NAME,
                        'stage_k': stage_k,
                        'out_k': out_k,
                        'citation_score_agg': score_agg,
                        'use_reranker': cfg.USE_RERANKER,
                    })

                    train_preds = [sweep_pipeline.predict(q) for q in train_df['query'].fillna('').tolist()]
                    val_preds = [sweep_pipeline.predict(q) for q in val_df['query'].fillna('').tolist()]
                    train_f1 = macro_f1(train_df['gold_citations'].fillna('').tolist(), train_preds)
                    val_f1 = macro_f1(val_df['gold_citations'].fillna('').tolist(), val_preds)

                    mlflow.log_metric('train_macro_f1', float(train_f1))
                    mlflow.log_metric('val_macro_f1', float(val_f1))

                    if best is None or val_f1 > best['val_macro_f1']:
                        best = {
                            'stage_k': stage_k,
                            'out_k': out_k,
                            'citation_score_agg': score_agg,
                            'train_macro_f1': float(train_f1),
                            'val_macro_f1': float(val_f1),
                        }

    print('Best sweep config:', best)
else:
    print('Skip sweep: set ENABLE_MLFLOW=True and install mlflow to track experiments.')

Skip sweep: set ENABLE_MLFLOW=True and install mlflow to track experiments.


In [34]:
# Build pipeline using the best sweep config if available; otherwise fallback to current cfg.

if cfg.ENABLE_MLFLOW and mlflow is not None:
    pipeline_cfg = PipelineConfig(
        mode='dense',
        dense_model_name=cfg.DENSE_MODEL_NAME,
        citation_score_agg=best['citation_score_agg'],
        stage_k=best['stage_k'],
        out_k=best['out_k'],
        use_reranker=cfg.USE_RERANKER,
        reranker_model_name=cfg.RERANKER_MODEL_NAME,
        reranker_top_n=cfg.RERANKER_TOP_N,
        reranker_batch_size=32,
    )

    pipeline = CitationRAGPipeline(
        corpus_df=corpus_df,
        config=pipeline_cfg,
        local_dense_model_path=cfg.LOCAL_DENSE_MODEL_PATH,
        local_reranker_model_path=cfg.LOCAL_RERANKER_MODEL_PATH,
    )

In [35]:
test_df = pd.read_csv(paths.test)
preds = [pipeline.predict(q) for q in test_df['query'].fillna('').tolist()]

submission_df = pd.DataFrame({
    'query_id': test_df['query_id'],
    'predicted_citations': preds,
})

submission_path = artifact_dir / cfg.SUBMISSION_NAME
submission_df.to_csv(submission_path, index=False)
print('Saved submission to:', submission_path)
submission_df.head(5)

Saved submission to: artifacts/submission.csv


,query_id,predicted_citations
0,test_001,Art. 10 Abs. 2 URG;Art. 62 Abs. 1 URG;Art. 13 Abs. 2 BV;Art. 17 GeoIG;Art. 59 MSchG;Art. 16 Abs. 2 GeoIG;Art. 129 Abs. 1 IPRG;Art. 36 De...
1,test_002,Art. 23 BVG;Art. 10 Abs. 1 UVG;Art. 37 Abs. 3 IVV;Art. 59 Abs. 1 SVG;Art. 9 Abs. 1 UVG;Art. 21 Abs. 4 ATSG;Art. 39 UVG;Art. 61 Abs. 1 SV...
2,test_003,4A_379/2016 E. 3.3.1;Art. 77 Abs. 1 OR;Art. 71 Abs. 1 SVG;Art. 4 UWG;Art. 205 Abs. 1 OR;Art. 76 Abs. 4 SVG;Art. 31 Abs. 1 OR;Art. 14 Abs...
3,test_004,Art. 309 ZPO;BGE 131 III 306 E. 3.1.2;BGE 132 III 342 E. 4.1;Art. 27 Abs. 2 BEG;Art. 83 Abs. 1 SchKG;BGE 132 III 564 E. 3.2.2;4A_42/2015...
4,test_005,4A_379/2016 E. 3.3.1;Art. 976 OR;Art. 175 Abs. 1 OR;Art. 176 Abs. 1 OR;Art. 265a Abs. 4 SchKG;Art. 725 Abs. 2 OR;Art. 7 Abs. 1 FIDLEG;Ar...


In [36]:
pipeline.dense.embedding

## Kaggle Submission Notes

1. Ensure all required Python packages are available in the Kaggle environment or bundled as datasets.
2. For strict offline runs:
   - Keep `MODE='dense'`.
   - If using reranker, set `LOCAL_RERANKER_MODEL_PATH`.
3. Keep runtime under 12 hours by controlling:
   - `STAGE_K`
   - chunking settings
   - reranker usage
4. The generated file for submission is `artifacts/submission.csv`.